In [2]:
import torch
import open_clip
import pandas as pd
from collections import defaultdict
from basemodel import load_finetuned_timm_wrapper 

In [3]:
def get_vit_architecture(model, model_name="ViT"):
    """
    Parses a Vision Transformer model and returns a structured list of its layers.
    This version is more robust to handle naming differences in patch embedding and MLP layers
    between DINO (timm) and OpenCLIP.
    """
    arch_list = []
    
    # --- Find the core ViT parts ---
    if hasattr(model, 'visual'): # Handle full OpenCLIP model
        vit_model = model.visual
    else:
        vit_model = model

    # --- Identify key components by type and name ---
    patch_embed = vit_model.patch_embed if hasattr(vit_model, 'patch_embed') else vit_model.conv1
    cls_token = vit_model.cls_token if hasattr(vit_model, 'cls_token') else vit_model.class_embedding
    pos_embed = vit_model.pos_embed if hasattr(vit_model, 'pos_embed') else vit_model.positional_embedding
    
    if hasattr(vit_model, 'blocks'):
        blocks = vit_model.blocks
    elif hasattr(vit_model, 'transformer'):
        blocks = vit_model.transformer.resblocks
    else:
        raise ValueError("Could not find transformer blocks.")

    # --- Build the structured list ---
    
    # CORRECTED: Handle patch embedding details
    if hasattr(patch_embed, 'proj'): # DINO style
        patch_embed_details = f"Output features: {patch_embed.proj.out_channels}"
    else: # OpenCLIP style (it's a raw Conv2d)
        patch_embed_details = f"Output features: {patch_embed.out_channels}"

    arch_list.append({
        "Component": "Input",
        "Layer Name": "patch_embed" if hasattr(vit_model, 'patch_embed') else "conv1",
        "Layer Type": type(patch_embed).__name__,
        "Details": patch_embed_details
    })
    
    arch_list.append({
        "Component": "Input",
        "Layer Name": "cls_token" if hasattr(vit_model, 'cls_token') else "class_embedding",
        "Layer Type": "Parameter",
        "Details": f"Shape: {cls_token.shape}"
    })
    
    arch_list.append({
        "Component": "Input",
        "Layer Name": "pos_embed" if hasattr(vit_model, 'pos_embed') else "positional_embedding",
        "Layer Type": "Parameter",
        "Details": f"Shape: {pos_embed.shape}"
    })
    
    if hasattr(vit_model, 'ln_pre'):
        arch_list.append({
            "Component": "Pre-Blocks",
            "Layer Name": "ln_pre",
            "Layer Type": type(vit_model.ln_pre).__name__,
            "Details": ""
        })
        
    for i, block in enumerate(blocks):
        block_prefix = f"Block {i}"
        
        norm1 = block.norm1 if hasattr(block, 'norm1') else block.ln_1
        attn = block.attn
        norm2 = block.norm2 if hasattr(block, 'norm2') else block.ln_2
        mlp = block.mlp
        
        arch_list.append({ "Component": block_prefix, "Layer Name": "norm1" if hasattr(block, 'norm1') else "ln_1", "Layer Type": type(norm1).__name__, "Details": "" })
        arch_list.append({ "Component": block_prefix, "Layer Name": "attn", "Layer Type": type(attn).__name__, "Details": f"Heads: {attn.num_heads}" })
        
        if hasattr(block, 'ls1'):
            arch_list.append({ "Component": block_prefix, "Layer Name": "ls1 (LayerScale)", "Layer Type": type(block.ls1).__name__, "Details": f"Dim: {block.ls1.gamma.shape[0]}" })
            
        arch_list.append({ "Component": block_prefix, "Layer Name": "norm2" if hasattr(block, 'norm2') else "ln_2", "Layer Type": type(norm2).__name__, "Details": "" })
        
        # CORRECTED: Handle MLP details
        if hasattr(mlp, 'fc2'): # DINO style
            mlp_details = f"Out Features: {mlp.fc2.out_features}"
        elif hasattr(mlp, 'c_proj'): # OpenCLIP style
            mlp_details = f"Out Features: {mlp.c_proj.out_features}"
        else:
            mlp_details = "N/A"
            
        arch_list.append({ "Component": block_prefix, "Layer Name": "mlp", "Layer Type": type(mlp).__name__, "Details": mlp_details })
        
        if hasattr(block, 'ls2'):
            arch_list.append({ "Component": block_prefix, "Layer Name": "ls2 (LayerScale)", "Layer Type": type(block.ls2).__name__, "Details": f"Dim: {block.ls2.gamma.shape[0]}" })

    final_norm = vit_model.norm if hasattr(vit_model, 'norm') else vit_model.ln_post
    arch_list.append({ "Component": "Output", "Layer Name": "norm" if hasattr(vit_model, 'norm') else "ln_post", "Layer Type": type(final_norm).__name__, "Details": "" })
    
    if hasattr(vit_model, 'proj'):
        arch_list.append({ "Component": "Output", "Layer Name": "proj", "Layer Type": "Parameter", "Details": f"Shape: {vit_model.proj.shape}" })

    return pd.DataFrame(arch_list)

In [4]:
CHECKPOINT_PATH = "/workspaces/vast-gorilla/gorillawatch/data/models/max_checkpoints/saved_checkpoints/giantbodybest74ens82.pth"
IMG_SIZE = 518
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BACKBONE = "vit_giant_patch14_dinov2.lvd142m"
EMBEDDING_DIM = 256 # The output size you trained for



model_wrapper, weights = load_finetuned_timm_wrapper(
    checkpoint_path=CHECKPOINT_PATH,
    backbone_name=BACKBONE,
    embedding_size=EMBEDDING_DIM,
    image_size=IMG_SIZE,
    device=DEVICE,
)

dino_model = model_wrapper.model

--- Loading Finetuned TimmWrapper Model ---


/workspaces/bachelor_thesis_code/src/bachelor_thesis/basemodel.py:202: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint_best = torch.load(checkpoint_path, map_locati

Building model architecture with backbone 'vit_giant_patch14_dinov2.lvd142m' and embedding size 256...


INFO:timm.models._builder:Loading pretrained weights from Hugging Face hub (timm/vit_giant_patch14_dinov2.lvd142m)
INFO:timm.models._hub:[timm/vit_giant_patch14_dinov2.lvd142m] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
INFO:basemodel:No pooling layer reset necessary.


State dict loading message: <All keys matched successfully>
Associated model transforms created successfully.
--- Model loading complete ---


In [5]:
clip_model_full, _, _ = open_clip.create_model_and_transforms('ViT-g-14', pretrained='laion2b_s34b_b88k')
clip_model = clip_model_full.visual

INFO:root:Loaded ViT-g-14 model config.
INFO:root:Loading pretrained ViT-g-14 weights (laion2b_s34b_b88k).


In [6]:
dino_df = get_vit_architecture(dino_model, model_name="DINO")
openclip_df = get_vit_architecture(clip_model, model_name="OpenCLIP")

In [7]:
def create_canonical_name(row):
    name = row['Layer Name'].split(' ')[0]
    return f"{row['Component']}::{name}"

In [8]:
dino_df['Canonical Name'] = dino_df.apply(create_canonical_name, axis=1)
openclip_df['Canonical Name'] = openclip_df.apply(create_canonical_name, axis=1)

# Merge the dataframes on the canonical name for a side-by-side view
comparison_df = pd.merge(
    dino_df,
    openclip_df,
    on="Canonical Name",
    how="outer",
    suffixes=('_DINO', '_OpenCLIP')
)

# Tidy up the final dataframe for display
comparison_df = comparison_df.set_index("Canonical Name")
display_cols = ['Layer Name_DINO', 'Layer Type_DINO', 'Layer Name_OpenCLIP', 'Layer Type_OpenCLIP']
comparison_df = comparison_df[display_cols]

with pd.option_context('display.max_rows', None, 'display.width', 120):
    print(comparison_df)

                              Layer Name_DINO Layer Type_DINO   Layer Name_OpenCLIP Layer Type_OpenCLIP
Canonical Name                                                                                         
Block 0::attn                            attn       Attention                  attn  MultiheadAttention
Block 0::ln_1                             NaN             NaN                  ln_1           LayerNorm
Block 0::ln_2                             NaN             NaN                  ln_2           LayerNorm
Block 0::ls1                 ls1 (LayerScale)      LayerScale                   NaN                 NaN
Block 0::ls2                 ls2 (LayerScale)      LayerScale                   NaN                 NaN
Block 0::mlp                              mlp          GluMlp                   mlp          Sequential
Block 0::norm1                          norm1       LayerNorm                   NaN                 NaN
Block 0::norm2                          norm2       LayerNorm   

In [10]:
vit_block = dino_model.blocks[0]

print("\n=== DINO Transformer Block[0] Structure Overview ===\n")

for name, module in vit_block.named_children():
    print(f"--- {name} ---")
    print(module)
    print()


=== DINO Transformer Block[0] Structure Overview ===

--- norm1 ---
LayerNorm((1536,), eps=1e-06, elementwise_affine=True)

--- attn ---
Attention(
  (qkv): Linear(in_features=1536, out_features=4608, bias=True)
  (q_norm): Identity()
  (k_norm): Identity()
  (attn_drop): Dropout(p=0.0, inplace=False)
  (proj): Linear(in_features=1536, out_features=1536, bias=True)
  (proj_drop): Dropout(p=0.0, inplace=False)
)

--- ls1 ---
LayerScale()

--- drop_path1 ---
Identity()

--- norm2 ---
LayerNorm((1536,), eps=1e-06, elementwise_affine=True)

--- mlp ---
GluMlp(
  (fc1): Linear(in_features=1536, out_features=8192, bias=True)
  (act): SiLU()
  (drop1): Dropout(p=0.0, inplace=False)
  (norm): Identity()
  (fc2): Linear(in_features=4096, out_features=1536, bias=True)
  (drop2): Dropout(p=0.0, inplace=False)
)

--- ls2 ---
LayerScale()

--- drop_path2 ---
Identity()



In [13]:
clip_block = clip_model.transformer.resblocks[0]

print("\n=== OPEN CLIP Transformer Block[0] Structure Overview ===\n")

for name, module in clip_block.named_children():
    print(f"--- {name} ---")
    print(module)
    print()


=== OPEN CLIP Transformer Block[0] Structure Overview ===

--- ln_1 ---
LayerNorm((1408,), eps=1e-05, elementwise_affine=True)

--- attn ---
MultiheadAttention(
  (out_proj): NonDynamicallyQuantizableLinear(in_features=1408, out_features=1408, bias=True)
)

--- ls_1 ---
Identity()

--- ln_2 ---
LayerNorm((1408,), eps=1e-05, elementwise_affine=True)

--- mlp ---
Sequential(
  (c_fc): Linear(in_features=1408, out_features=6144, bias=True)
  (gelu): GELU(approximate='none')
  (c_proj): Linear(in_features=6144, out_features=1408, bias=True)
)

--- ls_2 ---
Identity()

